In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

DATA_PATH = Path("bank-full.csv")
RANDOM_STATE = 42

plt.style.use("ggplot")
pd.set_option("display.max_columns", None)


In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("\nTarget counts:")
print(df["y"].value_counts())
print("\nTarget ratio:")
print(df["y"].value_counts(normalize=True).round(4))

df.head()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df["y"].value_counts().plot(kind="bar", ax=axes[0, 0], color=["#4C72B0", "#55A868"])
axes[0, 0].set_title("Target Distribution")

df["job"].value_counts().plot(kind="bar", ax=axes[0, 1], color="#4C72B0")
axes[0, 1].set_title("Job Distribution")
axes[0, 1].tick_params(axis="x", rotation=45)

df["education"].value_counts().plot(kind="bar", ax=axes[1, 0], color="#C44E52")
axes[1, 0].set_title("Education Distribution")

df["marital"].value_counts().plot(kind="bar", ax=axes[1, 1], color="#8172B2")
axes[1, 1].set_title("Marital Distribution")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["age"].hist(bins=20, ax=axes[0], color="#4C72B0", edgecolor="black")
axes[0].set_title("Age Distribution")

df["balance"].hist(bins=30, ax=axes[1], color="#55A868", edgecolor="black")
axes[1].set_title("Balance Distribution")

plt.tight_layout()
plt.show()


In [ ]:
X = df.drop(columns=["y", "duration"])
y = df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Duration excluded:", "duration" not in X.columns)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain target ratio:")
print(y_train.value_counts(normalize=True).round(4))
print("\nTest target ratio:")
print(y_test.value_counts(normalize=True).round(4))


In [ ]:
X_train_encoded = pd.get_dummies(X_train)
X_test_encoded = pd.get_dummies(X_test)
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join="left", axis=1, fill_value=0)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=1,
)

model.fit(X_train_encoded, y_train)
print("Model training completed.")


In [ ]:
y_pred = model.predict(X_test_encoded)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, pos_label='yes'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred, pos_label='yes'):.4f}")
print(f"F1: {f1_score(y_test, y_pred, pos_label='yes'):.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=["no", "yes"],
    cmap="Blues",
    ax=ax,
)
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()
